# Hedging research data feasibility
## tl;dr
Historical trade access is verified; a subsequent free historical quote probe also succeeded. Continue with a quote-first feasibility gate. A complete research panel and ML statistical adequacy remain unverified.
## Context & Methods
Provisional BTC/ETH, hourly rebalance, 24-hour same-contract holding diagnostic and desired 12 months. No fixed research specification or trained model. This notebook only reads saved raw evidence; it never fetches data or starts a collector.
### Key Assumptions
Positive-volume bars are a diagnostic, not a requirement for valid quotes. Grid alignment is not simultaneous price observation. The sampled contracts/hours are purposive and not representative. Overlapping holding windows are not independent observations.


In [1]:
from pathlib import Path
import json
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "option_collector").is_dir())
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from experiments.research_quality import analyze
metrics = analyze(ROOT)
print("Verified saved body/archive hashes and bounded input checks.")
print("Live requests in saved experiments:", sum(v["requests"] for v in metrics["live_probes"].values()))


Verified saved body/archive hashes and bounded input checks.
Live requests in saved experiments: 29


## Data
History experiment: 19 October 2026 Asia/Bangkok; cutoff 17 October 2026 23:59:59.999 UTC. Follow-up: 22 October 2026, 29 requests total. Raw bodies stay in ignored `experiments/results/`; metrics and complete methodology are in `docs/RESEARCH_FEASIBILITY.md`. Exact checks are in `experiments/research_quality.py`.
## Results
### 1. Candle density and 24-hour paths
Counts concern the two selected option contracts. Candidate windows overlap; endpoint trades do not establish an executable quote.


In [2]:
for row in metrics["hourly_profiles"]:
    if "PERPETUAL" in row["instrument"]: continue
    print(row["instrument"])
    print(f"  Zero-volume bars: {row['zero_volume']}/{row['bars']} ({row['zero_volume_fraction']:.2%})")
    print("  Longest zero-volume run (hours):", row["longest_zero_volume_hours"])
    print("  24h starts / positive-volume endpoints:", row["holding_24h_candidates"], "/", row["holding_24h_positive_volume_endpoints"])


BTC-30OCT26-95000-C
  Zero-volume bars: 1338/1909 (70.09%)
  Longest zero-volume run (hours): 40
  24h starts / positive-volume endpoints: 1885 / 272
ETH-25DEC26-3200-C
  Zero-volume bars: 4754/5001 (95.06%)
  Longest zero-volume run (hours): 259
  24h starts / positive-volume endpoints: 4977 / 24


### 2. Historical trade breadth and as-of risk
Each sample is one hour, options only. `has_more=false` is an API pagination result, not independent assurance that the archive has no missing trades. Last-trade age at 13:00 measures trade staleness, not quote staleness. Five minutes is a diagnostic reference.


In [3]:
for row in metrics["chain_hour_profiles"]:
    print(row["start_utc"], row["currency"])
    print("  Trades / instruments / expiries:", row["trades"], row["instruments"], row["expiries"])
    print("  Instruments with last trade older than 5 min:", row["last_trade_older_than_5min"], "/", row["instruments"])
    print("  Missing trade IV / mark / index:", row["missing_iv"], row["missing_mark"], row["missing_index"])
print("Sampled pagination covers sequences:", metrics["pagination_checked_sequences"])
print("Response sequence order monotonic:", metrics["pagination_response_sequence_sorted"])


2025-10-01T12:00:00+00:00 BTC
  Trades / instruments / expiries: 923 220 12
  Instruments with last trade older than 5 min: 197 / 220
  Missing trade IV / mark / index: 0 0 0
2025-10-01T12:00:00+00:00 ETH
  Trades / instruments / expiries: 528 152 11
  Instruments with last trade older than 5 min: 143 / 152
  Missing trade IV / mark / index: 0 0 0
2026-04-17T12:00:00+00:00 BTC
  Trades / instruments / expiries: 733 220 12
  Instruments with last trade older than 5 min: 182 / 220
  Missing trade IV / mark / index: 0 0 0
2026-04-17T12:00:00+00:00 ETH
  Trades / instruments / expiries: 334 122 11
  Instruments with last trade older than 5 min: 82 / 122
  Missing trade IV / mark / index: 0 0 0
Sampled pagination covers sequences: [1, 200]
Response sequence order monotonic: False


## Takeaways
- Historical trade prices, trade IV and event mark/index enable a trade-based pilot without waiting for prospective collection.
- Sparse candles and asynchronous trades require explicit as-of, freshness and valuation policies. Future nearest observations must not leak into features.
- Expired examples and two separated one-hour windows do not validate full-year chain completeness or ML sufficiency.
- Empirical spread/slippage requires historical quotes; assumed costs are scenarios. Funding/P&L units and the BSAD formula still need to be fixed.
- Updated next step: freeze the research contract, audit a complete free quote session and a traceable BS/BSAD ledger, then assess expansion; see docs/RESEARCH_DECISION.md.

Execution note: cells executed top-to-bottom in CPython with standard-library output capture; no native Jupyter kernel or nbformat validation was available. This file does not itself retrigger the live probes.


## Update: free historical quotes and project decision
Three disjoint one-minute Tardis ticker samples on 1 October 2025 returned 16 selected options plus BTC/ETH perpetuals. This is an availability/schema check, not a whole-day universe audit. Selection uses saved within-hour trades and is forward-looking for decisions inside that hour; later use needs an explicit prior-period liquidity rule and target population. Ticker observation age is not quote-change age or guaranteed execution.
Checks below are offline; exact code is in experiments/research_quote_quality.py. Source access terms and reconciled independent reviews are in docs/RESEARCH_DECISION.md.


In [4]:
from experiments.research_quote_quality import analyze as analyze_quotes
quote_metrics = analyze_quotes()
print("Saved free quote requests:", quote_metrics["requests"])
print("Ticker messages:", quote_metrics["messages"])
for row in quote_metrics["rows"]:
    print(row["from"], "symbols", row["symbols"], "invalid BBO", row["invalid_or_missing_bbo_messages"], "last valid", row["last_tickers_two_sided"])
print("Access verified; full-day input coverage and statistical sufficiency are not established.")


Saved free quote requests: 3
Ticker messages: 9748
2025-10-01T12:00:00.000Z symbols 18 invalid BBO 0 last valid 18
2025-10-01T13:00:00.000Z symbols 18 invalid BBO 0 last valid 18
2025-10-01T23:00:00.000Z symbols 18 invalid BBO 0 last valid 18
Access verified; full-day input coverage and statistical sufficiency are not established.
